In [0]:
%run "../libs/libs"

In [0]:
%run "../common/define_path"

In [0]:
otracking_path

In [0]:
def read_bronze_ordertracking():
    print('Reading the Bronze Table Data : ',end='')
    df_bronzeordertracking = (spark.readStream
                    .table(f"otracking.bronze.ordertracking")
                    )
    print(f'Reading otracking.bronze.ordertracking Success!')
    return df_bronzeordertracking

In [0]:
df_ordertracking = read_bronze_ordertracking()

In [0]:
from pyspark.sql.functions import col, to_date
df_ordertracking = df_ordertracking.select(col("trackingid"),
                               col("shipmentno"),
                               col("orderstatus"),
                               col("createdon").alias("trackingtime"))

                               

In [0]:
from pyspark.sql.functions import current_timestamp
df_ordertracking = convert_to_int(df_ordertracking, "trackingid")
df_ordertracking = df_ordertracking.withColumn("ingestion_time",current_timestamp())
df_ordertracking = convert_to_timestamp(df_ordertracking, "trackingtime")

In [0]:
def write_silver_ordertracking(StreamingDF):
      write_StreamSilver = (StreamingDF.writeStream
                .format('delta')
                .option('checkpointLocation',f'{otracking_path}/silver/checkpoint/ordertracking/')
                .outputMode('append')
                .queryName("silverordertrackingstream")
                .trigger(availableNow=True)
                .toTable("otracking.silver.ordertracking")
            )
      write_StreamSilver.awaitTermination()

In [0]:
write_silver_ordertracking(df_ordertracking)

In [0]:
%sql
select * from otracking.silver.ordertracking limit 1000